In [1]:
!pip install transformers==4.44.2 joblib==1.4.2 scikit-learn==1.6.0 numpy==1.26.4 pandas==2.2.3 scipy==1.13.1 seaborn==0.13.2 tqdm==4.66.5 lightgbm==4.5.0 xgboost==2.1.3

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 1.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 2.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.6/57.6 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.5/9.5 MB 72.2 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 301.8/301.8 kB 16.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.5/13.5 MB 99.8 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 38.6/38.6 MB 45.5 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 294.9/294.9 kB 14.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.4/78.4 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 153.9/153.9 MB 11.3 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 88.8 MB/s eta 0:00:00:00:01
  Attempting uninstall: tqdm
    Found existing installation: tqdm 4.67.1
    Un

In [2]:
import os
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
import numpy as np
from transformers import AutoTokenizer, AutoModelForSequenceClassification, AdamW
import torch
from torch.utils.data import DataLoader, Dataset


# Load datasets
train_df = pd.read_csv('/kaggle/input/dataset-caco2/Train_Caco2.csv')
train_df = train_df[['ID', 'SMILES', 'Permeability']]
test_df = pd.read_csv('/kaggle/input/dataset-caco2/Test_Caco2.csv')
test_df = test_df[['ID', 'SMILES', 'Permeability']]

In [3]:
tokenizer = AutoTokenizer.from_pretrained("seyonec/PubChem10M_SMILES_BPE_450k")
model = AutoModelForSequenceClassification.from_pretrained('seyonec/PubChem10M_SMILES_BPE_450k', num_labels=1)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

tokenizer_config.json:   0%|          | 0.00/62.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/515 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/772 [00:00<?, ?B/s]

/usr/local/lib/python3.11/dist-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


pytorch_model.bin:   0%|          | 0.00/336M [00:00<?, ?B/s]

Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at seyonec/PubChem10M_SMILES_BPE_450k and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [4]:
# dataset class
class SMILESDataset(Dataset):
    def __init__(self, dataframe, tokenizer, max_length=325):
        self.tokenizer = tokenizer
        self.dataframe = dataframe
        self.max_length = max_length

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, idx):
        smiles = self.dataframe.iloc[idx]['SMILES']
        permeability = self.dataframe.iloc[idx]['Permeability']
        inputs = self.tokenizer(smiles, return_tensors='pt', padding="max_length", truncation=True, max_length=self.max_length)
        
        input_ids = inputs['input_ids'].squeeze(0)  # Shape: (sequence_length,)
        attention_mask = inputs['attention_mask'].squeeze(0)  # Shape: (sequence_length,)
        
        return {
            'input_ids': input_ids,
            'attention_mask': attention_mask,
            'labels': torch.tensor(permeability, dtype=torch.float)
        }

In [5]:
# datasets
train_dataset = SMILESDataset(train_df, tokenizer)
test_dataset = SMILESDataset(test_df, tokenizer)
batch_size = 16
# data loaders
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

In [6]:
optimizer = torch.optim.AdamW(model.parameters(), lr=5e-5)
num_epochs = 20

In [7]:
# Training loop
from tqdm import tqdm
for epoch in range(num_epochs):
    print(f"Entered Epoch {epoch + 1}")
    model.train()
    train_loss = 0

    for batch in tqdm(train_loader, desc=f'Training Epoch {epoch + 1}/{num_epochs}', unit='batch'):
        optimizer.zero_grad()

        # Move all batch tensors to device
        batch = {k: v.to(device) for k, v in batch.items()}
        labels = batch["labels"].unsqueeze(1)  # still shape: (batch_size, 1)

        # Forward pass
        outputs = model(
            input_ids=batch["input_ids"],
            attention_mask=batch["attention_mask"],
            labels=labels
        )
        loss = outputs.loss
        train_loss += loss.item()

        # Backprop and optimizer step
        loss.backward()
        optimizer.step()

    avg_train_loss = train_loss / len(train_loader)
    print(f'Epoch {epoch + 1}/{num_epochs} - Train Loss: {avg_train_loss:.4f}')

Entered Epoch 1


Training Epoch 1/20: 100%|██████████| 63/63 [00:28<00:00,  2.25batch/s]


Epoch 1/20 - Train Loss: 2.1915
Entered Epoch 2


Training Epoch 2/20: 100%|██████████| 63/63 [00:28<00:00,  2.21batch/s]


Epoch 2/20 - Train Loss: 0.4808
Entered Epoch 3


...

Epoch 18/20 - Train Loss: 0.0959
Entered Epoch 19


Training Epoch 19/20: 100%|██████████| 63/63 [00:32<00:00,  1.93batch/s]


Epoch 19/20 - Train Loss: 0.0943
Entered Epoch 20


Training Epoch 20/20: 100%|██████████| 63/63 [00:32<00:00,  1.93batch/s]

Epoch 20/20 - Train Loss: 0.0975


In [8]:
# Saving the model after training
model_name = 'PubChem10M_SMILES_BPE_450k_model_1_caco2'
model_save_path = f'/kaggle/working/{model_name}'
os.makedirs(model_save_path, exist_ok=True)

tokenizer.save_pretrained(model_save_path)
model.save_pretrained(model_save_path)

print(f'Model and tokenizer saved to {model_save_path}')

Model and tokenizer saved to /kaggle/working/PubChem10M_SMILES_BPE_450k_model_1_caco2


In [9]:
from scipy.stats import pearsonr, spearmanr

model.eval()
test_loss = 0
test_true_labels = []
predictions = []

with torch.no_grad():
    for batch in tqdm(test_loader, desc='Testing', unit='batch'):
      
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].unsqueeze(1).to(device).float()

        outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
        loss = outputs.loss
        test_loss += loss.item()

        test_true_labels.extend(labels.cpu().numpy())
        preds = outputs.logits.squeeze().cpu().numpy()  
        predictions.extend(preds)

# Final test loss
avg_test_loss = test_loss / len(test_loader)
print(f'Test Loss: {avg_test_loss:.4f}')

test_true_labels = np.array(test_true_labels).flatten()
predictions = np.array(predictions)
print(test_true_labels.shape)
print(predictions.shape)

# Performance metrics
mse = mean_squared_error(test_true_labels, predictions)
rmse = np.sqrt(mse)
mae = mean_absolute_error(test_true_labels, predictions)
r2 = r2_score(test_true_labels, predictions)
PCC,_ = pearsonr(test_true_labels, predictions)
SCC,_ = spearmanr(test_true_labels, predictions)

# Print performance metrics
print(f'Mean Squared Error: {mse:.4f}')
print(f'Root Mean Squared Error: {rmse:.4f}')
print(f'Mean Absolute Error: {mae:.4f}')
print(f'R^2 Score: {r2:.4f}')
print(f'Pearson Correlation Coefficient: {PCC:.4f}')
print(f'Spearman Correlation Coefficient: {SCC:.4f}')

# Print hyperparameters
print("Hyperparameters:")
print(f"Learning Rate: {5e-5}")
print(f"Batch Size: 16")
print(f"Epochs: {num_epochs}")

Testing: 100%|██████████| 16/16 [00:02<00:00,  5.58batch/s]


Test Loss: 0.3456
(252,)
(252,)
Mean Squared Error: 0.3488
Root Mean Squared Error: 0.5906
Mean Absolute Error: 0.4416
R^2 Score: 0.4402
Pearson Correlation Coefficient: 0.7601
Spearman Correlation Coefficient: 0.7300
Hyperparameters:
Learning Rate: 5e-05
Batch Size: 16
Epochs: 20


In [10]:
import os
import pandas as pd
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from transformers import AutoTokenizer, AutoModel

from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

model_name = 'PubChem10M_SMILES_BPE_450k_model_1_caco2'
model_save_path = f'/kaggle/working/{model_name}'

if not os.path.exists(model_save_path):
    raise FileNotFoundError(f"The model directory {model_save_path} does not exist.")

# Load tokenizer and model
tokenizer = AutoTokenizer.from_pretrained(model_save_path, trust_remote_code=True)
model = AutoModel.from_pretrained(model_save_path, trust_remote_code=True).to(device)

Some weights of RobertaModel were not initialized from the model checkpoint at /kaggle/working/PubChem10M_SMILES_BPE_450k_model_1_caco2 and are newly initialized: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [11]:
# Load your datasets
train_df = pd.read_csv('/kaggle/input/dataset-caco2/Train_Caco2.csv')
train_df = train_df[['ID', 'SMILES', 'Permeability']]
test_df = pd.read_csv('/kaggle/input/dataset-caco2/Test_Caco2.csv')
test_df = test_df[['ID', 'SMILES', 'Permeability']]

In [12]:
train_encodings = tokenizer(list(train_df['SMILES']), truncation=True, padding=True, max_length=325, return_tensors="pt")
test_encodings = tokenizer(list(test_df['SMILES']), truncation=True, padding=True, max_length=325, return_tensors="pt")

In [13]:
from tqdm import tqdm 
batch_size = 16 

def generate_embeddings(encodings, batch_size):
    embeddings = []
    model.eval() 
    with torch.no_grad():
        for i in tqdm(range(0, len(encodings['input_ids']), batch_size), desc="Processing batches"):
            batch = {key: val[i:i + batch_size].to(device) for key, val in encodings.items()}  
            outputs = model(**batch)
            embeddings.append(outputs.last_hidden_state)
    return torch.cat(embeddings, dim=0)


In [14]:
train_embeddings = generate_embeddings(train_encodings, batch_size)
print(train_embeddings.shape)
train_embeddings = torch.mean(train_embeddings, dim=1)
print(train_embeddings.shape)

Processing batches: 100%|██████████| 63/63 [00:07<00:00,  8.16it/s]


torch.Size([1007, 220, 768])
torch.Size([1007, 768])


In [15]:
column_names = [f'x_fine_emb_pubchem{i}' for i in range(train_embeddings.shape[1])]
embeddings_df = pd.DataFrame(data=train_embeddings.cpu().numpy(), columns=column_names)
train_data = pd.concat([train_df, embeddings_df], axis=1)

In [16]:
test_embeddings = generate_embeddings(test_encodings, batch_size)
print(test_embeddings.shape)
test_embeddings = torch.mean(test_embeddings, dim=1)
print(test_embeddings.shape)

Processing batches: 100%|██████████| 16/16 [00:01<00:00,  8.89it/s]

torch.Size([252, 222, 768])
torch.Size([252, 768])


In [17]:
column_names = [f'x_fine_emb_pubchem{i}' for i in range(test_embeddings.shape[1])]
embeddings_df = pd.DataFrame(data=test_embeddings.cpu().numpy(), columns=column_names)
test_data = pd.concat([test_df, embeddings_df], axis=1)

In [18]:
train_data.to_csv("/kaggle/working/Train_PubChem10M_SMILES_BPE_450k_model_1_fine_tuned_embeddings_caco2.csv",index=False)
test_data.to_csv("/kaggle/working/Test_PubChem10M_SMILES_BPE_450k_model_1_fine_tuned_embeddings_caco2.csv",index=False)

In [19]:
import numpy as np
import pandas as pd
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor, AdaBoostRegressor, ExtraTreesRegressor
from sklearn.linear_model import LinearRegression, LogisticRegression  # LogisticRegression is not used for regression
from sklearn.neighbors import KNeighborsRegressor
from sklearn.svm import SVR
from sklearn.neural_network import MLPRegressor
import xgboost as xgb
import lightgbm as lgb
from sklearn.model_selection import KFold, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from scipy.stats import pearsonr, spearmanr
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt

In [20]:
train_data = pd.read_csv("/kaggle/working/Train_PubChem10M_SMILES_BPE_450k_model_1_fine_tuned_embeddings_caco2.csv")
test_data = pd.read_csv("/kaggle/working/Test_PubChem10M_SMILES_BPE_450k_model_1_fine_tuned_embeddings_caco2.csv")

In [21]:
def train_and_test_predict(models, X_train, y_train, X_test, y_test):
    kf = KFold(n_splits=5, shuffle=True, random_state=42)
    results = {}
    predictions = []  

    for model in models:
        model_name = model.__class__.__name__
        predictions_train = []
        actual_y_train = []

        test_predictions_folds = []

        

        for train_index, val_index in kf.split(X_train):
            X_train_fold, X_val_fold = X_train.iloc[train_index], X_train.iloc[val_index]
            y_train_fold, y_val_fold = y_train.iloc[train_index], y_train.iloc[val_index]

            model.fit(X_train_fold, y_train_fold)

            y_pred_fold = model.predict(X_val_fold)
            y_pred_fold = np.clip(y_pred_fold, -10, -3.4)
            predictions_train.extend(y_pred_fold)
            actual_y_train.extend(y_val_fold)

            predictions_test_fold = model.predict(X_test)
            predictions_test_fold = np.clip(predictions_test_fold, -10, -3.4)
            test_predictions_folds.append(predictions_test_fold)


        mse_train = mean_squared_error(actual_y_train, predictions_train)
        mae_train = mean_absolute_error(actual_y_train, predictions_train)
        rmse_train = np.sqrt(mse_train)
        r2_train = r2_score(actual_y_train, predictions_train)
        pearson_train, _ = pearsonr(actual_y_train, predictions_train)
        spearman_train, _ = spearmanr(actual_y_train, predictions_train)


        predictions_test_mean = np.mean(test_predictions_folds, axis=0)
        predictions_test_std = np.std(test_predictions_folds, axis=0)

        mse_test = mean_squared_error(y_test, predictions_test_mean)
        mae_test = mean_absolute_error(y_test, predictions_test_mean)
        rmse_test = np.sqrt(mse_test)
        r2_test = r2_score(y_test, predictions_test_mean)
        print(r2_test)
        pearson_test, _ = pearsonr(y_test, predictions_test_mean)
        spearman_test, _ = spearmanr(y_test, predictions_test_mean)
        
        

        predictions.append({
            'Model': model_name,
            'Y Train pred': predictions_train,
            'Y Test actual': y_test,
            'Test prediction folds': test_predictions_folds,
            'Test Predictions Mean': predictions_test_mean,
            'Test Predictions Std': predictions_test_std,

        })

        results[model_name] = {
            'Train MSE (5 fold cv)': f"{mse_train:.4f}",
            'Train MAE (5 fold cv)': f"{mae_train:.4f}",
            'Train RMSE (5 fold cv)': f"{rmse_train:.4f}",
            'Train RMSE (5 fold cv)': f"{rmse_train:.4f}",
            'Train R2 (5 fold cv)': f"{r2_train:.4f}",
            'Train PCC (5 fold cv)': f"{pearson_train:.4f}",
            'Train SCC (5 fold cv)': f"{spearman_train:.4f}",
            'Test MSE': f"{mse_test:.4f}",
            'Test MAE': f"{mae_test:.4f}",
            'Test RMSE': f"{rmse_test:.4f}",
            'Test R2': f"{r2_test:.4f}",
            'Test Pearson Correlation': f"{pearson_test:.4f}",
            'Test Spearman Correlation': f"{spearman_test:.4f}",
        }

    results_df = pd.DataFrame(results).T
    predictions_df = pd.DataFrame(predictions)

    return results_df, predictions_df



In [22]:
X_train = train_data.drop(['ID','SMILES','Permeability'],axis=1)
y_train = train_data['Permeability']
print("X_train shape: ",X_train.shape)
print("y_train shape: ",y_train.shape)
print("XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX")

X_test = test_data.drop(['ID','SMILES','Permeability'],axis=1)
y_test = test_data['Permeability']
print("X_test shape: ",X_test.shape)
print("y_test shape: ",y_test.shape)
print("XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX")
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
X_train = pd.DataFrame(X_train_scaled, columns=X_train.columns, index=X_train.index)
X_test = pd.DataFrame(X_test_scaled, columns=X_test.columns, index=X_test.index)
models = [
    lgb.LGBMRegressor(objective='regression',metric='rmse',boosting_type='gbdt',num_leaves=31,learning_rate=0.05,random_state=42),
    DecisionTreeRegressor(random_state=42),
    RandomForestRegressor(n_jobs=-1, random_state=42),
    GradientBoostingRegressor(random_state=42),
    AdaBoostRegressor(random_state=42),
    xgb.XGBRegressor(random_state=42),
    ExtraTreesRegressor(n_jobs=-1, n_estimators=100, random_state=42),
    LinearRegression(), 
    KNeighborsRegressor(n_neighbors=3),
    SVR(),  
    MLPRegressor(random_state=42)
]
result_df, prediction_df = train_and_test_predict(models, X_train,y_train, X_test,  y_test)
result_df

X_train shape:  (1007, 768)
y_train shape:  (1007,)
XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
X_test shape:  (252, 768)
y_test shape:  (252,)
XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.008943 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 195840
[LightGBM] [Info] Number of data points in the train set: 805, number of used features: 768
[LightGBM] [Info] Start training from score -6.179395
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No furth

,Train MSE (5 fold cv),Train MAE (5 fold cv),Train RMSE (5 fold cv),Train R2 (5 fold cv),Train PCC (5 fold cv),Train SCC (5 fold cv),Test MSE,Test MAE,Test RMSE,Test R2,Test Pearson Correlation,Test Spearman Correlation
LGBMRegressor,0.0546,0.1766,0.2336,0.9149,0.9566,0.9505,0.2519,0.3734,0.5019,0.5957,0.7793,0.7389
DecisionTreeRegressor,0.1266,0.2717,0.3558,0.8025,0.9001,0.8851,0.2608,0.3830,0.5107,0.5815,0.7705,0.7253
RandomForestRegressor,0.0576,0.1824,0.2400,0.9101,0.9546,0.9477,0.2492,0.3709,0.4992,0.6000,0.7798,0.7392
GradientBoostingRegressor,0.0564,0.1816,0.2376,0.9119,0.9550,0.9493,0.2523,0.3738,0.5023,0.5951,0.7788,0.7369
AdaBoostRegressor,0.0659,0.1979,0.2566,0.8972,0.9474,0.9384,0.2588,0.3835,0.5087,0.5847,0.7701,0.7247
XGBRegressor,0.0660,0.1968,0.2569,0.8971,0.9472,0.9379,0.2532,0.3739,0.5032,0.5937,0.7780,0.7391
ExtraTreesRegressor,0.0560,0.1778,0.2367,0.9126,0.9557,0.9498,0.2495,0.3705,0.4995,0.5996,0.7801,0.7390
LinearRegression,2.2584,1.1398,1.5028,-2.5239,0.3998,0.4403,0.9470,0.7444,0.9731,-0.5198,0.4434,0.4216
KNeighborsRegressor,0.0799,0.2185,0.2826,0.8753,0.9362,0.9234,0.2401,0.3566,0.4900,0.6146,0.7925,0.7600
SVR,0.0527,0.1688,0.2296,0.9178,0.9584,0.9589,0.2514,0.3742,0.5014,0.5966,0.7760,0.7377


In [23]:
result_df

,Train MSE (5 fold cv),Train MAE (5 fold cv),Train RMSE (5 fold cv),Train R2 (5 fold cv),Train PCC (5 fold cv),Train SCC (5 fold cv),Test MSE,Test MAE,Test RMSE,Test R2,Test Pearson Correlation,Test Spearman Correlation
LGBMRegressor,0.0546,0.1766,0.2336,0.9149,0.9566,0.9505,0.2519,0.3734,0.5019,0.5957,0.7793,0.7389
DecisionTreeRegressor,0.1266,0.2717,0.3558,0.8025,0.9001,0.8851,0.2608,0.3830,0.5107,0.5815,0.7705,0.7253
RandomForestRegressor,0.0576,0.1824,0.2400,0.9101,0.9546,0.9477,0.2492,0.3709,0.4992,0.6000,0.7798,0.7392
GradientBoostingRegressor,0.0564,0.1816,0.2376,0.9119,0.9550,0.9493,0.2523,0.3738,0.5023,0.5951,0.7788,0.7369
AdaBoostRegressor,0.0659,0.1979,0.2566,0.8972,0.9474,0.9384,0.2588,0.3835,0.5087,0.5847,0.7701,0.7247
XGBRegressor,0.0660,0.1968,0.2569,0.8971,0.9472,0.9379,0.2532,0.3739,0.5032,0.5937,0.7780,0.7391
ExtraTreesRegressor,0.0560,0.1778,0.2367,0.9126,0.9557,0.9498,0.2495,0.3705,0.4995,0.5996,0.7801,0.7390
LinearRegression,2.2584,1.1398,1.5028,-2.5239,0.3998,0.4403,0.9470,0.7444,0.9731,-0.5198,0.4434,0.4216
KNeighborsRegressor,0.0799,0.2185,0.2826,0.8753,0.9362,0.9234,0.2401,0.3566,0.4900,0.6146,0.7925,0.7600
SVR,0.0527,0.1688,0.2296,0.9178,0.9584,0.9589,0.2514,0.3742,0.5014,0.5966,0.7760,0.7377


In [24]:
prediction_df

,Model,Y Train pred,Y Test actual,Test prediction folds,Test Predictions Mean,Test Predictions Std
0,LGBMRegressor,"[-7.106395599269513, -7.099529166698216, -5.81...",0 -7.22 1 -6.21 2 -7.24 3 -5.9...,"[[-6.459790883064867, -6.156882303777009, -5.9...","[-6.319637332901738, -6.071805095257862, -5.96...","[0.08047294010663404, 0.06717755053771575, 0.0..."
1,DecisionTreeRegressor,"[-7.1, -7.03, -5.739999999999999, -6.07, -5.91...",0 -7.22 1 -6.21 2 -7.24 3 -5.9...,"[[-6.64, -6.08, -6.21, -5.866461092, -5.919999...","[-6.276000000000001, -6.136, -6.04, -5.8672922...","[0.3382661673889365, 0.2044113499784198, 0.106..."
2,RandomForestRegressor,"[-6.999697611209999, -7.141272219359996, -5.74...",0 -7.22 1 -6.21 2 -7.24 3 -5.9...,"[[-6.464878318099997, -6.050516372879997, -5.9...","[-6.375168388809998, -5.966897347895999, -5.91...","[0.054462844395461105, 0.051748563719587276, 0..."
3,GradientBoostingRegressor,"[-7.045104998501785, -7.178916245620542, -5.94...",0 -7.22 1 -6.21 2 -7.24 3 -5.9...,"[[-6.393802146858201, -6.162266957876556, -6.0...","[-6.377526273785909, -6.053868012921595, -5.95...","[0.09108171361922378, 0.09064129947642235, 0.0..."
4,AdaBoostRegressor,"[-6.893227448414631, -7.190947390171782, -5.68...",0 -7.22 1 -6.21 2 -7.24 3 -5.9...,"[[-6.450130516874129, -5.89805524351207, -5.89...","[-6.408848335013781, -6.005179161248794, -5.80...","[0.08629751730477628, 0.07120000024687391, 0.0..."
5,XGBRegressor,"[-7.0289583, -7.257389, -5.7935343, -5.4829254...",0 -7.22 1 -6.21 2 -7.24 3 -5.9...,"[[-6.376616, -6.2224836, -5.889052, -5.9116397...","[-6.271144, -6.014147, -5.9783916, -5.830362, ...","[0.08961115, 0.18830717, 0.056298874, 0.071115..."
6,ExtraTreesRegressor,"[-6.984126458879996, -7.054699999999997, -5.72...",0 -7.22 1 -6.21 2 -7.24 3 -5.9...,"[[-6.36759067803, -5.996808933780001, -5.87781...","[-6.381438509198, -5.967557383215999, -5.87127...","[0.013149102522137687, 0.03905249462572731, 0...."
7,LinearRegression,"[-5.890614712906214, -6.693288257706414, -7.74...",0 -7.22 1 -6.21 2 -7.24 3 -5.9...,"[[-4.745937156408005, -9.577828609039685, -6.7...","[-5.125949055835534, -7.154330650406135, -6.42...","[0.36797354820269945, 1.8508649039565852, 0.54..."
8,KNeighborsRegressor,"[-7.14, -7.066666666666666, -5.893333333333333...",0 -7.22 1 -6.21 2 -7.24 3 -5.9...,"[[-6.386666666666667, -6.003333333333333, -5.9...","[-6.397333333333334, -5.971666666666667, -5.97...","[0.0322765824984271, 0.03617856946990681, 0.05..."
9,SVR,"[-6.927401713553152, -7.120729871864434, -5.84...",0 -7.22 1 -6.21 2 -7.24 3 -5.9...,"[[-6.315369774861486, -6.039269429480895, -5.9...","[-6.288133317437113, -6.037350382486125, -5.96...","[0.03541804445518052, 0.020462360515780896, 0...."


In [25]:
result_df.to_csv('/kaggle/working/Results_PubChem10M_SMILES_BPE_450k_model_1_fine_tuned_embeddings_caco2.csv')
prediction_df.to_csv('/kaggle/working/Prediction_data_PubChem10M_SMILES_BPE_450k_model_1_fine_tuned_embeddings_caco2.csv')